# FADE — VLM Extraction Demo

**Runtime required:** GPU (T4 or better)  
In Colab: `Runtime → Change runtime type → T4 GPU`

**What this notebook does:**
1. Installs dependencies
2. **Section A** — Zero-shot inference test (~10 min)
3. **Section B** — Short QLoRA training proof-of-concept (~30–60 min on T4, ~15 min on A100)

## Setup

In [ ]:
# Clone the repo
import os

if not os.path.exists('ai-document-processing'):
    !git clone https://github.com/juweriya1/ai-document-processing.git

%cd ai-document-processing
!git pull origin main

In [ ]:
# Install dependencies (takes ~3-5 min)
!pip install -q \
    transformers>=4.45.0 \
    torch torchvision \
    bitsandbytes>=0.43.0 \
    peft>=0.11.0 \
    accelerate \
    qwen-vl-utils>=0.0.8 \
    datasets>=2.18.0 \
    Pillow \
    trl

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Section A — Zero-shot Inference (~10 min)

Loads Qwen2-VL-7B-Instruct in 4-bit and runs extraction on a CORD receipt sample.  
No training needed — this shows what the model can do out of the box.

In [ ]:
# Load one CORD receipt sample
from datasets import load_dataset
import json

print("Downloading one CORD test sample...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="test")

SAMPLE_INDEX = 0  # change this to try different receipts
row = dataset[SAMPLE_INDEX]
pil_image = row["image"]

# Parse ground truth
gt_raw = json.loads(row["ground_truth"])["gt_parse"]
ground_truth = {
    "vendor_name":  gt_raw.get("store_info", {}).get("store_name"),
    "total_amount": gt_raw.get("total", {}).get("total_price"),
    "subtotal":     gt_raw.get("total", {}).get("subtotal_price"),
    "tax":          gt_raw.get("total", {}).get("tax_price"),
}

print(f"Image size: {pil_image.size}")
print("\nGround truth:")
for k, v in ground_truth.items():
    if v: print(f"  {k}: {v}")

pil_image  # display the receipt in the notebook

In [ ]:
# Load the VLM extractor and run inference
# First run: downloads ~15 GB model weights from HuggingFace (~3-5 min)
import sys
sys.path.insert(0, '.')

import time
from src.backend.extraction.vlm_extractor import VLMExtractor

print("Loading VLMExtractor (downloads model on first run)...")
t0 = time.time()
extractor = VLMExtractor()
_ = extractor.model  # trigger load now so timing is accurate
print(f"Model loaded in {time.time()-t0:.0f}s")

print("\nRunning inference...")
t1 = time.time()
fields, line_items = extractor.extract_from_image(pil_image.convert("RGB"))
print(f"Done in {time.time()-t1:.1f}s")

In [ ]:
# Print results
print("=" * 55)
print("EXTRACTED FIELDS")
print("=" * 55)
if fields:
    for f in fields:
        print(f"  {f['field_name']:<20} {str(f['field_value']):<25}  conf={f['confidence']:.2f}")
else:
    print("  (no fields extracted)")

if line_items:
    print("\nLINE ITEMS")
    print("=" * 55)
    for item in line_items:
        print(f"  {item.get('description',''):<30} qty={item.get('quantity',0)}  total={item.get('total',0)}")

print("\nCOMPARISON vs GROUND TRUTH")
print("=" * 55)
pred_map = {f["field_name"]: f["field_value"] for f in fields}
for field, gold in ground_truth.items():
    if not gold:
        continue
    pred = pred_map.get(field, "(missing)")
    match = "✓" if str(gold).strip() == str(pred).strip() else "✗"
    print(f"  {match} {field:<20} gold={gold}  pred={pred}")

---
## Section B — QLoRA Training Proof-of-Concept

Runs **100 steps** just to confirm training works end-to-end.  
**This is not the full training run** — do the full 500-step run on the A100 MIG.

| Hardware | 100 steps | 500 steps (full) |
|----------|-----------|------------------|
| T4 (Colab free) | ~45 min | ~3.5 hr |
| A100 (Colab Pro / MIG) | ~15 min | ~1 hr |

**Skip this section if you just wanted to test inference above.**

In [ ]:
# Optional: free up VRAM from inference before training
import torch, gc
del extractor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved")

In [ ]:
# Run 100-step proof-of-concept training
# Saves adapter weights to adapters/qlora_poc/
!python scripts/train_qlora.py \
    --max_steps 100 \
    --output_dir adapters/qlora_poc

print("\nDone. Adapter saved to adapters/qlora_poc/")

In [ ]:
# Test the fine-tuned adapter on the same sample
from src.backend.extraction.vlm_extractor import VLMExtractor

print("Loading fine-tuned extractor...")
ft_extractor = VLMExtractor(adapter_path="adapters/qlora_poc")

fields_ft, _ = ft_extractor.extract_from_image(pil_image.convert("RGB"))

print("\nFINE-TUNED vs ZERO-SHOT vs GROUND TRUTH")
print("=" * 65)
pred_zs  = {f["field_name"]: f["field_value"] for f in fields}
pred_ft  = {f["field_name"]: f["field_value"] for f in fields_ft}
for field, gold in ground_truth.items():
    if not gold:
        continue
    zs  = pred_zs.get(field, "(missing)")
    ft  = pred_ft.get(field, "(missing)")
    print(f"  {field:<20} gold={gold}")
    print(f"    zero-shot : {zs}")
    print(f"    fine-tuned: {ft}")

In [ ]:
# Optional: save adapter to Google Drive so it persists after Colab disconnects
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r adapters/qlora_poc /content/drive/MyDrive/fade_adapters/